# Reduction and Analysis of JWST Imaging Data

Table of Contents
- [Introduction](#Introduction)
- [Downloading Data from JWST](#Downloading-Data-from-JWST)
- [Installation](#Installation)
- [Running The Pipeline](#Running-The-Pipeline)
- [Stage 0 to 3](#Stage-0-to-3)
- [Output Files](#Output-Files)



# Introduction

## Downloading Data from JWST


In this repository data was downloaded using the dedicated Search MAST for JWST(https://mast.stsci.edu/search/ui/#/jwst) interface.

MAST(The Mikulski Archive for Space Telescope) serves as the official repository that hosts all the publically available JWST science data.The archive stores observations from all four scientific instruments onboard JWST.NIRcam, NIRSpec, MIRI and NIRISS.  

This archive allows observations to be filtered according to parameters such as target,program/proposal ID, instrument, observing mode, detector and filter.


<img src="/home/vibhak16/jwst_project/Images/image2.png" width="1000">

Using the JWST search interface, the search was specified for JWST observations from Program ID 1345 corresponding to 
Cosmic Evolution Early Release Science Survey(CEERS). The data type was selected to IMAGE as we analyse the NIRCam imaging data.NIRCam was selected as the instrument because the CEERS observations used in this analysis were obtained with JWST's Near Infrared Camera.

<img src="/home/vibhak16/jwst_project/Images/image1.png" width="1000">

At the product selection stage, the level 1b data product was selected because it contains the uncalibrated exposure data requried as the starting point for detector-level calibration. 
This includes the _uncal.fits files which were selected as the aim of the analysis was to perform the JWST Stage 1(calwebb_detector1) calibration pipeline manually and examine the individual detector level correction steps.

Level 1b: Uncalibrated detector data (_uncal.fits), containing the raw exposure readings and used as input to the Detector1 pipeline.

Level 2a: Calibrated exposure level data (_cal.fits), produced after detector-level and instrument specific calibration through the Stage 2 pipeline.

Level 3: Processed science products(_i2d.fits) formed by combining multiple dithers and multiple calibrated exposure.

Preview/thumbnail products: Lower resolution images provided mainly for quick visualization and inspection, not for scientific processing.


<img src="/home/vibhak16/jwst_project/Images/image3.png" width="1000">

Each datadet has ten level 1b uncalibrated NIRCam fits files. Each file corresponds to different  detectors and dither positions which are from multiple exposure. 
The selected uncalibrated exposure are used as inputs to the Detector1 pipeline, with one of the ten downloaded fits file(jw01345004001_08201_00003_nrcb1_uncal.fits) processed initially.
In the file jw01345004001_08201_00003_nrcb1_uncal.fits
01345	Program/Proposal ID — Program 1345 for CEERS survey

004 → Observation number

001 → Visit number

_08201_ → Exposure identifier within the observation/visit\

_00003_	→ Exposure/segment identifier

nrcb1 → NIRCam detector B1

uncal → uncalibrated data

.fits → FITS astronomical data file


## Installation

Installing Python Environment,the JWST science calibration pipeline is a Python based software. For all the python packages to be fully functioning we need a seperate Conda environment specifially for JWST data reduction. It provides an isolated environment in which the all the scientific packages are valid. The JWST calibration pipeline was installed and executed within the WSL environment using Ubuntu based Linux distribution. Within WSL, Conda was created and managed as an isolated Python environment dedicated to the JWST data reduction pipeline.Using a terminal window,new Conda environment was created and the pipeline was installed.



First, an isolated environment was created with Python 3.13


```bash
conda create -n jwst_env python=3.13
```

The newly created environment was then activated

```bash
conda activate jwst_env
```

Finally, install jwst into the environment

```bash
pip install jwst

 Set up the Calibration Reference Data System(CRDS)

Before processing the JWST observations with the Calibration Pipeline, the Calibration Reference Data System(CRDS) was configured. CRDS is used by the pipeline to manage and provide the calibration reference files requried during data processing.These files consists of instrument and detector specific calibration information that are standad parameter information requried by individual pipeline steps. Examples include dark current reference files, superbias reference files, linearity files,saturation reference files etc.

```bash
JWST Calibration Pipeline Version: 2.0.0
CRDS - INFO -  Calibration SW Found: jwst 2.0.0 (/home/vibhak16/anaconda3/envs/jwst_env/lib/python3.12/site-packages/jwst-2.0.0.dist-info)
Using CRDS Context: jwst_1535.pmap

The configuration was therefore established in the WSL terminal using

```bash 
export CRDS_SERVER_URL = https://jwst-crds.stsci.edu
export CRDS_PATH = /home/vibhak16/data/crds_cache

The CRDS context specifies the environment variable mapped to the reference files used for the observations,


```bash
export CRDS_CONTEXT=jwst_1535.pmap

## Running the Pipeline

The JWST NIRCam data was processed using the JWST Science Calibration Pipeline.The redcution of the data was performed through three stages with pipelines specified for each stage. Detector1, Image2 and Image3 for each stages and the outputs from each stages served as the input for the subsequent stage.

### Detector1 Pipeline

The first stage of processing was performed using the Detector1 Pipeline. The uncalibrated NIRCam science image of calibration Level 1.b(_uncal.fits) file was used as the input to this stage.
Detector1 performs detector level corrections including data-quality initialization, saturation correction, superbias correction, reference pixel correction, linearity correction, dark current correction, jump detection and ramp fitting.The input file "jw01345004001_08201_00003_nrcb1_uncal.fits" was called and intermediate result files were saved along with the _rate.fits file which is obtained after the ramp fitting step.

```text
*_uncal.fits
      ↓
Detector1Pipeline
      ↓
Stage 1 products
      ↓
*_rate.fits
```

The Detector1 pipeline and its individual calibration steps are first imported into the notebook. The pipeline is executed using Detector1Pipeline.call().

```bash
from jwst.pipeline import Detector1Pipeline
from jwst.dq_init import DQInitStep
from jwst.saturation import SaturationStep
from jwst.refpix import RefPixStep
from jwst.superbias import SuperBiasStep
from jwst.jump import JumpStep
from jwst.linearity import LinearityStep
from jwst.dark_current import DarkCurrentStep
from jwst.ramp_fitting import RampFitStep

The above import statements loads the Detector1 pipeline and its associated classes from the installed JWST Science Clibration Pipeline package into the Python environment. Each imported class above represents one individual specific level of calibration step. These import initialize the access to the corresponding pipeline and when called it executes the said functions.

The complete code used to execute the Detector1 pipeline is provided in the following notebook
"GIVE THE LINK OF SECOND JUPYTER NOTEBOOK"

### Calling The Pipeline

```bash
from jwst.pipeline import Detector1Pipeline
import time

uncal_file = '/home/vibhak16/jwst_project/my_ceers_data/Obs001/uncal/jw01345004001_08201_00003_nrcb1_uncal.fits'

print("Running full Detector1Pipeline with intermediate results")
start = time.time()

result = Detector1Pipeline.call(uncal_file,
                                save_results=True,
                                output_dir='/home/vibhak16/jwst_project/my_ceers_data/Obs001/Stage1/',
                                steps={
                                    'dq_init':       {'save_results': True},
                                    'saturation':    {'save_results': True},
                                    'superbias':     {'save_results': True},
                                    'refpix':        {'save_results': True},
                                    'linearity':     {'save_results': True},
                                    'dark_current':  {'save_results': True},
                                    'jump':          {'save_results': True},
                                    'ramp_fit':      {'save_results': True}
                                })

end = time.time()

The code executes the complete Stage 1 Detector1 pipeline on the specified uncalibrated JWST NIRCam exposure. First, Detector1Pipeline is imported from the JWST pipeline package, while the time module is imported to measure the execution time. The variable uncal_file specifies the path to the input _uncal.fits file, which contains the uncalibrated detector data. The Detector1Pipeline.call() command then starts the calibration, with "save_results=True" instructing the pipeline to save the final pipeline output and output_dir specifying the directory in which the products are stored. The steps dictionary explicitly enables saving the output after each specified Detector1 calibration steps, dq_init, saturation, superbias, refpix, linearity, dark_current, jump, and ramp_fit, allowing the intermediate products to be examined individually.

 The returned result contains the final pipeline result, while "start = time.time()" and "end = time.time()" record the times before and after execution, allowing the total processing time to be calculated.

## Level 0 to Level 3



```text
Level 0
Raw telemetry
    ↓
Level 1b
Uncalibrated detector data
(*_uncal.fits)
    ↓
Stage 1
Detector-level calibration
(*_rate.fits)
    ↓
Stage 2
Image-level calibration
(*_cal.fits)
    ↓
Stage 3 
Dithering
(*_i2d.fits)

### Stage 0

The raw telemetry received from the JWST spacecraft is converted to Stage 0 uncalibrated data by the Science Data Processing(SDP) system. In the initial process important information about the observation is added to the FITS header. This information comess from sources of the spacecraft's onboard telemetry. The "_uncal.fits" files produced by the SDP are obtained from MAST and are used as starting point for next steps of calibration.

### Stage 1

The Calwebb_detector1 pipeline constitutes Stage 1 of the JWST Science Calibration Pipeline. A number of detector level corrections are performed.We are working with Imaging data from NIRCam. It is applied to one exposure at a time, starting with an uncalibrated multi-accum ramp(_uncal.fits) where ramps-to-slopes processing takes place.

Each input raw data file consists of one more ramps(integrations) which contains increasing count values of detector readouts which is non-destructive as it does not resets the count after every integration.

The Detector1 pipeline performs detector-level calibration on the uncalibrated (_uncal.fits) data. It corrects and flags various detector effects, including data-quality issues, saturation, bias, reference-pixel effects, detector nonlinearity, dark current, and sudden events such as cosmic-ray hits. After these corrections, the ramp_fit step fits the up-the-ramp measurements for each pixel to determine its signal accumulation rate. The resulting pixel-by-pixel count rates are assembled into a two-dimensional (_rate.fits) image, which serves as the input for the Stage 2 Image2 pipeline.



## Output 

The output is a corrected but uncalibrated countrate which is also the slope image. The ramp_fit step obtains the count rate for each detector pixel by fitting a model using the least squares method to the pixel measurements. The fitted slopes for all the pixels are put together into a 2D image where each pixel value represents the rate at which the signal was accumulated, represented as Digital Numbers/second(DN/s).

After the ramp-fitting process, the slope determined for each individual detector pixel is placed back at the same spatial position that the pixel had on the detector. In this way, the individual slope values are assembled together to form a two-dimensional count-rate image. Each position in this 2D image corresponds to one physical pixel on the detector, and the numerical value stored at that position represents the rate at which signal was accumulated by that pixel during the exposure. Therefore, instead of containing a sequence of measurements for each pixel at different readout times, as in the original _uncal.fits data, the resulting image contains a single representative count-rate value for each pixel.

This two-dimensional count-rate image is written as the *_rate.fits product, which is the primary science output of the Detector1 stage. The values in this product describe the detector signal in terms of a rate, rather than the total accumulated detector signal. Pixels receiving more astronomical light will generally have larger count-rate values, while pixels receiving less light will have smaller values. Thus, the _rate.fits file can now be treated as an image of the detector's measured signal rate, but it is not yet the final calibrated astronomical image. Further image level corrections, such as flat-fielding, WCS assignment are performed in Stage 2 using the _rate.fits product as input.


<img src="/home/vibhak16/jwst_project/Images/uncalpic.jpg" width="500">

The above is the uncalibrated picture of jw01345004001_08201_00003_nrcb1_uncal.fits FITS file.It is a 4d data cube the first 2 dimensions are the column and row axes of the detector, while the 3rd dimension is determined by the number of groups per integration, and the 4th dimension by the number of integrations in the exposure

<img src="/home/vibhak16/jwst_project/Images/rateoutput.png" width="500">

This is rate.fits file,the output produced after ramp fitting in Stage 1. It contains a 2D count-rate image, where each pixel represents the measured count rate in counts per second obtained from the up-the-ramp data. 